# Fase 1 — VQR com AngleEmbedding

**Arquitetura:** 6 qubits | 4 camadas StronglyEntanglingLayers | AngleEmbedding (RY) | ⟨Z₀⊗Z₁⊗Z₂⊗Z₃⊗Z₄⊗Z₅⟩

**Nota:** R² negativo é esperado — o circuito raso sem re-uploading não consegue capturar a dinâmica epidêmica. A Fase 2 resolve isso.

In [1]:
try:
    import mlflow, mlflow.sklearn
    mlflow.set_tracking_uri("mlruns")
    _MLFLOW = False  # tracking desativado (entregável)
except ImportError:
    _MLFLOW = False
    print("[AVISO] mlflow nao instalado — execute: pip install mlflow")
import warnings; warnings.filterwarnings("ignore")
import os, json, sys, time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
from feature_engineering import construir_features, splits_validacao

CACHE = os.path.join(REPO_ROOT, "data", "dados_dengue_df_real.json")
with open(CACHE, encoding="utf-8") as f:
    dados_brutos = json.load(f)

dataset = construir_features(dados_brutos, n_lags=4)
splits  = splits_validacao(dataset)

NOMES = {0: ("C1", "Transmissão normal/crescente  (out/2023 – out/2024)"),
         1: ("C2", "Pico recorde 25.714 casos/sem  (jun/2024 – jun/2025)"),
         2: ("C3", "Pós-surto, Rt < 1              (out/2024 – jun/2025)")}

CENARIOS = {}
for idx, split in enumerate(splits[:3]):
    nome, desc = NOMES[idx]
    tr, te = split["treino"], split["teste"]
    CENARIOS[nome] = {
        "X_train": np.array(tr["X"]), "y_train": np.array(tr["y"]),
        "X_test":  np.array(te["X"]), "y_test":  np.array(te["y"]),
        "datas":   te.get("datas", []),
        "nome": desc,
        "periodo_treino": split.get("periodo_treino", ""),
        "periodo_teste":  split.get("periodo_teste",  ""),
    }

print(f"Features ({len(dataset['feature_names'])}): {dataset['feature_names']}")
for nome, d in CENARIOS.items():
    print(f"{nome}: treino={len(d['X_train'])} | teste={len(d['X_test'])} | "
          f"target_max={max(d['y_test']):.0f}")

# ── utilitários compartilhados (utils_qml.py na raiz do projeto) ─────────────
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(".."))
from utils_qml import (calcular_wis, metricas, salvar_padrao, plot_pred,
                        validar_json_saida, validar_pipeline,
                        testar_invariancia_quantica, testar_propriedades,
                        validar_golden)
testar_propriedades()
print("[OK] utils_qml importado")

c:\Users\julia\anaconda3\envs\qml_dengue\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Features (13): ['casos_est_lag1', 'casos_est_lag2', 'casos_est_lag3', 'casos_est_lag4', 'Rt_lag1', 'Rt_lag2', 'p_rt1_lag1', 'receptivo_lag1', 'transmissao_lag1', 'tempmed_lag1', 'umidmed_lag1', 'SE_sin', 'SE_cos']
C1: treino=36 | teste=143 | target_max=25714
C2: treino=88 | teste=91 | target_max=25714
C3: treino=125 | teste=54 | target_max=947
[HYPOTHESIS] Biblioteca nao instalada. Executando versao simplificada.
             Para instalar: pip install hypothesis
[PROP OK] 500 combinacoes aleatorias: WIS>=0, RMSE>=0, MAE>=0 em todos.
[OK] utils_qml importado


In [2]:
# ── Validação do pipeline de dados (integração) ──────────────────────────────
validar_pipeline(dataset, splits)


[PIPELINE OK] 13 features | 4 splits | serie=179 semanas | cenarios C1/C2/C3 prontos


True

In [3]:
import pennylane as qml
from pennylane import numpy as pnp
from copy import deepcopy
from sklearn.preprocessing import MinMaxScaler
print(f"PennyLane: {qml.__version__} | NumPy: {np.__version__}")

PennyLane: 0.40.0 | NumPy: 1.26.4


In [4]:
CONFIG = {
    "n_qubits": 6, "n_layers": 4, "n_features": 6,
    "n_epochs": 80, "lr": 0.01, "n_bootstrap": 5,
    "seed": 42, "patience": 15,
}
np.random.seed(CONFIG["seed"])

# Prepara dados: AngleEmbedding usa apenas os 4 primeiros features → normaliza para [0, π]
CENARIOS_F1 = {}
for cen, d in CENARIOS.items():
    sc = MinMaxScaler(feature_range=(0.01, 3.14))
    X_tr = sc.fit_transform(d["X_train"][:, :CONFIG["n_features"]])
    X_te = sc.transform(d["X_test"][:, :CONFIG["n_features"]])
    CENARIOS_F1[cen] = {**d, "X_train": X_tr, "X_test": X_te, "scaler": sc}
print("[OK] Dados preparados para AngleEmbedding (6 features, [0,π])")

[OK] Dados preparados para AngleEmbedding (6 features, [0,π])


In [5]:
try:
    _dev_name = "lightning.qubit"
    import pennylane_lightning  # noqa
except ImportError:
    _dev_name = "default.qubit"
    print("[AVISO] pennylane-lightning nao instalado, usando default.qubit (mais lento)")
n_q = CONFIG["n_qubits"]
dev = qml.device(_dev_name, wires=n_q)

@qml.qnode(dev)
def vqr_f1(x, weights):
    """AngleEmbedding (RY) + StronglyEntanglingLayers + medição ⟨Z₀⊗Z₁⊗Z₂⊗Z₃⟩."""
    qml.AngleEmbedding(x[:n_q], wires=range(n_q), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_q))
    return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1) @ qml.PauliZ(2) @ qml.PauliZ(3) @ qml.PauliZ(4) @ qml.PauliZ(5))

n_params = CONFIG["n_layers"] * n_q * 3
print(f"Circuito: {n_q} qubits | {CONFIG['n_layers']} camadas | {n_params} parâmetros")

rng_viz = np.random.RandomState(0)
w_viz = pnp.array(rng_viz.uniform(-np.pi, np.pi, (CONFIG["n_layers"], n_q, 3)))
x_viz = pnp.array(rng_viz.uniform(0, np.pi, n_q))
fig, _ = qml.draw_mpl(vqr_f1)(x_viz, w_viz)
plt.title("VQR Fase 1 — AngleEmbedding + StronglyEntanglingLayers")
plt.tight_layout(); plt.show()

Circuito: 6 qubits | 4 camadas | 72 parâmetros


In [6]:
# ── Invariância quântica (determinismo, bounds, shape) ──────────────────
testar_invariancia_quantica(
    circuit_fn=vqr_f1,
    n_qubits=CONFIG.get('n_qubits', 6),
    x_sample=x_viz,
    weights_sample=w_viz,
    contexto="Fase1_VQR_AngleEmbedding"
)

[QUANTICO OK] [Fase1_VQR_AngleEmbedding] Determinismo / bounds [-1,1] / shape(1) — OK


True

In [7]:
def treinar_f1(X_tr, y_tr, X_te, cfg=CONFIG):
    n_q, n_l, B = cfg["n_qubits"], cfg["n_layers"], cfg["n_bootstrap"]
    preds_mat = np.zeros((B, len(X_te)))
    loss_hist = []
    y_log = np.log1p(y_tr).astype(np.float64)
    mu_y, sig_y = y_log.mean(), y_log.std() + 1e-8
    y_norm = (y_log - mu_y) / sig_y

    for b in range(B):
        t0  = time.time()
        rng = np.random.RandomState(cfg["seed"] + b)
        idx = rng.choice(len(X_tr), size=len(X_tr), replace=True)
        Xb  = pnp.array(X_tr[idx], requires_grad=False)
        yb  = pnp.array(y_norm[idx], requires_grad=False)

        W = pnp.array(rng.uniform(-np.pi, np.pi, (n_l, n_q, 3)) * 0.01, requires_grad=True)
        sc = pnp.array(rng.uniform(0.5, 1.5), requires_grad=True)
        bi = pnp.array(0.0, requires_grad=True)
        opt = qml.AdamOptimizer(stepsize=cfg["lr"])

        best_loss, best_W, patience, hist = float("inf"), deepcopy(W.numpy()), 0, []
        for epoch in range(cfg["n_epochs"]):
            def cost(W_, sc_, bi_):
                preds = pnp.array([vqr_f1(Xb[i], W_) * sc_ + bi_ for i in range(len(Xb))])
                return pnp.mean((preds - yb) ** 2)
            (W, sc, bi), loss = opt.step_and_cost(cost, W, sc, bi)
            lv = float(loss); hist.append(lv)
            if lv < best_loss: best_loss, best_W, patience = lv, deepcopy(W.numpy()), 0
            else:
                patience += 1
                if patience >= cfg["patience"]: break

        W_b = pnp.array(best_W, requires_grad=False)
        raw = np.array([float(vqr_f1(pnp.array(X_te[i], requires_grad=False), W_b))
                        * float(sc) + float(bi) for i in range(len(X_te))])
        preds_mat[b] = np.maximum(np.expm1(raw * sig_y + mu_y), 0)
        loss_hist.append(hist)
        print(f"  réplica {b+1}/{B}: {len(hist)} épocas | loss={best_loss:.4f} | {time.time()-t0:.0f}s")
    return preds_mat, loss_hist

print("[OK] Função de treinamento F1 definida")

[OK] Função de treinamento F1 definida


In [8]:
RESULTADOS = {}
for cen, dados in CENARIOS_F1.items():
    print(f"\n{'='*60}\n  CENÁRIO {cen}\n{'='*60}")
    t_start = time.time()
    preds_matrix, loss_hist = treinar_f1(dados["X_train"], dados["y_train"], dados["X_test"])
    med = np.median(preds_matrix, axis=0)
    m   = metricas(dados["y_test"], med, preds_matrix, nome=f"VQR_F1_{cen}")
    RESULTADOS[cen] = {**m, "preds_matrix": preds_matrix, "mediana": med,
                        "y_test": dados["y_test"], "loss_hist": loss_hist,
                        "tempo_s": time.time() - t_start}
    print(f"  R²={m['R2']:.4f} | WIS={m['WIS']:.2f} | {RESULTADOS[cen]['tempo_s']/60:.1f} min")


  CENÁRIO C1
  réplica 1/5: 80 épocas | loss=0.5234 | 82s
  réplica 2/5: 80 épocas | loss=0.3030 | 75s
  réplica 3/5: 80 épocas | loss=0.3155 | 69s
  réplica 4/5: 80 épocas | loss=0.3127 | 91s
  réplica 5/5: 80 épocas | loss=0.2099 | 92s
  R²=-0.0361 | WIS=2167.60 | 6.8 min

  CENÁRIO C2
  réplica 1/5: 80 épocas | loss=0.4098 | 206s
  réplica 2/5: 80 épocas | loss=0.1807 | 217s
  réplica 3/5: 80 épocas | loss=0.4457 | 191s
  réplica 4/5: 80 épocas | loss=0.4580 | 218s
  réplica 5/5: 80 épocas | loss=0.3451 | 217s
  R²=-0.1471 | WIS=3330.95 | 17.5 min

  CENÁRIO C3
  réplica 1/5: 80 épocas | loss=0.6607 | 303s
  réplica 2/5: 80 épocas | loss=0.5888 | 294s
  réplica 3/5: 80 épocas | loss=0.5687 | 294s
  réplica 4/5: 80 épocas | loss=0.7010 | 296s
  réplica 5/5: 80 épocas | loss=0.6500 | 296s
  R²=-15.0591 | WIS=622.91 | 24.7 min


In [9]:
print(f"\n{'='*70}")
print(f"{'FASE 1 — VQR AngleEmbedding (6q, 4L, sem re-uploading)':^70}")
print(f"{'='*70}")
print(f"{'Cenário':<10} {'R²':>8} {'RMSE':>10} {'WIS':>10} {'WIS_norm':>10}")
print("-" * 70)
for cen, r in RESULTADOS.items():
    print(f"{cen:<10} {r['R2']:>8.4f} {r['RMSE']:>10.1f} {r['WIS']:>10.2f} {r['WIS_norm']:>10.4f}")
print("=" * 70)
print("Nota: R² negativo esperado — Fase 2 usa re-uploading para superar.")

plot_pred(RESULTADOS, "Fase 1 — VQR AngleEmbedding: Predição vs. Observado", "fase1_pred_vs_obs.png")

# Convergência
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (cen, r) in enumerate(RESULTADOS.items()):
    ax = axes[i]
    for h in r["loss_hist"]: ax.plot(h, color="lightgray", alpha=0.4, lw=0.8)
    mat = np.full((len(r["loss_hist"]), max(len(h) for h in r["loss_hist"])), np.nan)
    for j, h in enumerate(r["loss_hist"]): mat[j, :len(h)] = h
    ax.plot(np.nanmean(mat, axis=0), "tab:purple", lw=2, label="Média")
    ax.set_title(f"Cenário {cen}"); ax.set_xlabel("Época"); ax.set_yscale("log"); ax.grid(alpha=0.3)
plt.suptitle("Fase 1 — Convergência (5 réplicas)", fontweight="bold")
plt.tight_layout(); plt.savefig("fase1_convergencia.png", dpi=150, bbox_inches="tight"); plt.show()
OUTFILE = "fase1_resultados.json"


        FASE 1 — VQR AngleEmbedding (6q, 4L, sem re-uploading)        
Cenário          R²       RMSE        WIS   WIS_norm
----------------------------------------------------------------------
C1          -0.0361     5825.4    2167.60     0.7668
C2          -0.1471     7429.9    3330.95     0.8499
C3         -15.0591      757.5     622.91     1.1710
Nota: R² negativo esperado — Fase 2 usa re-uploading para superar.
[SALVO] fase1_pred_vs_obs.png


In [10]:
import json as _json, os as _os

## Justificativa dos Hiperparâmetros - VQR AngleEmbedding

| Hiperparâmetro | Valor | Justificativa | Referência |
|---|---|---|---|
| `n_qubits` | 6 | 2⁶ = 64 dimensões de Hilbert; dentro do regime NISQ prático; mais qubits que 4 sem custo proibitivo em simulação clássica | Preskill (2018). *Quantum Computing in the NISQ Era and Beyond*. Quantum, 2, 79 |
| `n_layers` | 4 | Circuito raso proposital (Fase 1 é baseline): sem re-uploading, camadas adicionais introduzem barren plateaus sem ganho expressivo | Cerezo et al. (2021). *Variational Quantum Algorithms*. Nat. Rev. Phys., 3, 625–644 |
| `n_features` | 6 | AngleEmbedding mapeia 1 feature por qubit via RY(xᵢ); usa os 6 primeiros lags de casos_est e Rt, as features com maior importância RF | Schuld & Petruccione (2021). *Machine Learning with Quantum Computers*. Springer |
| `lr` | 0.01 | Taxa de aprendizado conservadora para Adam sem annealing; compatível com circuito raso (sem re-uploading, sem custo de barren plateau) | Kingma & Ba (2015). *Adam: A Method for Stochastic Optimization*. ICLR |
| `n_bootstrap` | 5 | Mínimo para estimativa confiável do WIS por intervalo (Bracher et al., 2021) | Bracher et al. (2021). PLOS Comput. Biol. |
| `patience` | 15 | Early stopping: 15 épocas sem melhoria evita overfitting sem interromper exploração do espaço de parâmetros | Prechelt (1998). *Early Stopping — But When?* In: Neural Networks: Tricks of the Trade |

> **Papel desta fase:** serve como *lower bound* quântico — a ausência de re-uploading é intencional para quantificar seu impacto na Fase 2.

In [11]:
# 6 qubits × 4 camadas × 3 params + escala + bias = 74
SCHEMA_INFO = {
    "algoritmo": "VQR-AngleEmbedding",
    "fase": 1,
    "tipo": "variacional",
    "n_parametros_quanticos": CONFIG["n_layers"] * CONFIG["n_qubits"] * 3 + CONFIG["n_qubits"] + 1,
    "config": CONFIG,
}
doc = salvar_padrao(RESULTADOS, SCHEMA_INFO)
validar_json_saida(doc, contexto="Fase1_VQR_AngleEmbedding")
validar_golden(doc, contexto="Fase1_VQR_AngleEmbedding")
# ── MLflow: registro automático do experimento ────────────────────────────────
_MLFLOW = False  # tracking desativado (entregável)
if _MLFLOW:
    with mlflow.start_run(run_name="Fase1_VQR_AngleEmbedding"):
        mlflow.log_params(SCHEMA_INFO.get("config", {}))
        mlflow.log_param("algoritmo",  SCHEMA_INFO.get("algoritmo", ""))
        mlflow.log_param("fase",       SCHEMA_INFO.get("fase", 0))
        mlflow.log_param("tipo",       SCHEMA_INFO.get("tipo", ""))
        for _cen in ["C1", "C2", "C3"]:
            if _cen in doc:
                mlflow.log_metric(f"WIS_{_cen}",      doc[_cen].get("WIS", float("nan")))
                mlflow.log_metric(f"WIS_norm_{_cen}", doc[_cen].get("WIS_norm", float("nan")))
                mlflow.log_metric(f"R2_{_cen}",       doc[_cen].get("R2", float("nan")))
                mlflow.log_metric(f"RMSE_{_cen}",     doc[_cen].get("RMSE", float("nan")))


[PADRAO] fase01_vqr-angleembedding_resultados.json
  Algoritmo : VQR-AngleEmbedding
  Tipo      : variacional
  Parametros quanticos: 79
  C1: R2=-0.0361 | WIS=2167.60 | 409.0s
  C2: R2=-0.1471 | WIS=3330.95 | 1048.1s
  C3: R2=-15.0591 | WIS=622.91 | 1482.6s
[CONTRATO OK] [Fase1_VQR_AngleEmbedding] JSON valido — todos os campos e invariantes corretos
[GOLDEN OK] [Fase1_VQR_AngleEmbedding] Resultados dentro da tolerancia vs. referencia.
